# 3. Compare modalities

Load the saved subspace/transformation results, inspect controlled component stability, compare subspaces and covariance models, and assess transformation generalization. Expensive computations run in the printed Python commands; plotting remains editable in this notebook.


In [ ]:
from pathlib import Path
import sys
import json
import shlex
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
MEG_KIND = 'paired_coverage'
BLOCK_SCALING = True
N_COMPONENTS = 10
MODELS = ['PCA', 'PLSSVD', 'PCA_JOINT']
STABILITY_MODEL = 'PLSSVD'
SHUFFLE = 'TRIAL_ONLY'  # 'TRIAL_ONLY', 'PAIRING_ONLY', or 'BOTH'
COMPONENT = 1
N_ITERATIONS = 100
N_SPLITS = 5
SEED = 2026
RIDGE_ALPHA = .01
MAX_GRAM_GIB = 2.
CACHE_DIR = ROOT / 'out' / 'trial_cache'
ROI_LEVEL = 'region'
ROI_ERROR = 'sd_across_iterations'  # Or 'mean_channel_sd'.
ATLAS_IMAGE = None  # Optional local atlas; otherwise use the existing AAL workflow.
ATLAS_LABELS = None
MODEL_NAMES = {'PCA':'separate_pca', 'PLSSVD':'plssvd', 'PCA_JOINT':'joint_pca'}
SHUFFLE_NAMES = {'TRIAL_ONLY':'trials', 'PAIRING_ONLY':'pairing', 'BOTH':'both'}
MODEL_COLORS = {'separate_pca':'#555555', 'plssvd':'#009E73', 'joint_pca':'#CC79A7'}
SCALING_MODE = 'equal_variance' if BLOCK_SCALING else 'none'

sys.path[:0] = [str(ROOT), str(ROOT.parent), str(ROOT.parent/'LB')]
from vs3_0_compare_subspace import load_results
from vs3_0_weight_stability import load_weight_summary
from vs1_coverage_figures import apply_style, MODALITY_COLORS
apply_style()


## Load selected model results

`MODELS` accepts one or more models. Existing filenames and the `compare_subspace` analysis module are retained for compatibility. Saved MEG construction, scaling, and component count are checked before plotting. Choose a new output directory when changing fit settings.

In [ ]:
INPUT_DIR = ROOT / 'out' / 'compare_subspace' / MEG_KIND
OUTPUT_DIR = INPUT_DIR
RECOMPUTE = False

selected_models = [MODEL_NAMES[name.upper()] for name in MODELS]
command = ['python','-u',str(ROOT/'vs3_0_compare_subspace.py'),'--cache-dir',str(CACHE_DIR),
    '--output-dir',str(OUTPUT_DIR),'--meg-kind',MEG_KIND,'--models',*selected_models,
    '--dimensions',*[str(k) for k in range(1,N_COMPONENTS+1)],'--n-splits',str(N_SPLITS),
    '--block-scaling',SCALING_MODE,'--seed',str(SEED),'--ridge-alpha',str(RIDGE_ALPHA),
    '--max-gram-gib',str(MAX_GRAM_GIB)]
print(shlex.join(command))
tables = None
if not RECOMPUTE and (INPUT_DIR/'COMPLETE.json').exists():
    config, loaded = load_results(INPUT_DIR)
    if (config['meg_kind'] != MEG_KIND or config['block_scaling'] != SCALING_MODE
        or N_COMPONENTS not in config['dimensions'] or not set(selected_models) <= set(config['models'])):
        print('Saved settings differ. Select matching results or change OUTPUT_DIR before running.')
    else:
        tables = {name: frame[frame.model.isin(selected_models)].copy() if 'model' in frame else frame
                  for name,frame in loaded.items()}
        display(pd.Series(config, name='Saved comparison settings'))
if tables is None:
    print('Run the command in a new output directory if needed, then load the completed results.')


## Component and time-course stability

Select one `COMPONENT`, one `STABILITY_MODEL`, and one `SHUFFLE` scheme. Trial-only changes the selected trial half; pairing-only changes subject association while retaining the trial half; both changes the two factors using the corresponding controlled random streams. When a MEG construction does not depend on subject association, pairing-only variation need not change its fitted signals. For `random_control`, changing the matching seed also changes the random source selection.

Components are matched to iteration zero using temporal scores. Coupled models share the permutation and sign across modalities; separate PCA is aligned independently within each modality. Weight heatmaps correlate magnitudes across fixed electrode coverage slots. Time-course heatmaps correlate signed, aligned temporal scores across time points. Pairwise entries share iterations and are not independent replicates.

ROI plots show participant-balanced mean absolute weight, with SD across iterations of the ROI mean (or the selected mean channel SD). Signed weights are correlated **across iterations first**, then grouped by bilateral iEEG–MEG ROI pairs. The last row shows the mean and SD across valid channel-pair correlations; that SD measures pair heterogeneity, not uncertainty across iterations. Constant trajectories are undefined and remain gray. Both modalities retain the fixed iEEG ROI labels and coverage-slot order, even when native MEG sources change.


In [ ]:
INPUT_DIR = ROOT / 'out' / 'weight_stability' / MEG_KIND
OUTPUT_DIR = INPUT_DIR
RECOMPUTE = False

variation = SHUFFLE_NAMES[SHUFFLE]
stability_model = MODEL_NAMES[STABILITY_MODEL.upper()]
stability_fit_models = list(dict.fromkeys([*selected_models, stability_model]))
command = ['python','-u',str(ROOT/'vs3_0_weight_stability.py'),'--cache-dir',str(CACHE_DIR),
    '--output-dir',str(OUTPUT_DIR),'--meg-kind',MEG_KIND,'--models',*stability_fit_models,
    '--conditions',variation,'--iterations',str(N_ITERATIONS),'--components',str(N_COMPONENTS),
    '--block-scaling',SCALING_MODE,'--seed',str(SEED),'--roi-level',ROI_LEVEL,
    '--max-gram-gib',str(MAX_GRAM_GIB)]
if ATLAS_IMAGE is not None: command += ['--atlas-image',str(ATLAS_IMAGE)]
if ATLAS_LABELS is not None: command += ['--atlas-labels',str(ATLAS_LABELS)]
print('Fit and summarize:',shlex.join(command))
print('Summarize existing fits:',shlex.join(command+['--summarize-only']))
stability = None
if not RECOMPUTE:
    try:
        stability = load_weight_summary(INPUT_DIR/variation, model=stability_model, component=COMPONENT,
            meg_kind=MEG_KIND, block_scaling=SCALING_MODE, n_components=N_COMPONENTS)
        if stability['config']['variation'] != variation:
            raise ValueError('Saved shuffle scheme differs from SHUFFLE.')
        if stability['summary']['roi_level'] != ROI_LEVEL:
            stability = None
            print('ROI level differs; rerun the summary command.')
        else:
            print('Saved stability iterations:',stability['config']['iterations'])
    except (FileNotFoundError, ValueError) as error:
        stability = None
        print(error)
if stability is None:
    print('Run the required command, set RECOMPUTE=False, and rerun this cell. Use a new directory for changed fit settings.')


In [ ]:
FIGSIZE = (17, 16)
HEATMAP_CMAP = 'RdBu_r'
ROI_LABEL_SIZE = 8

if stability is not None:
    fig, axes = plt.subplots(4, 2, figsize=FIGSIZE, constrained_layout=True)
    n = stability['config']['iterations']
    cmap = plt.get_cmap(HEATMAP_CMAP).copy(); cmap.set_bad('lightgray')
    for row, (table, metric, title) in enumerate([
            ('iteration_correlations','magnitude_r','Weight magnitude stability'),
            ('timecourse_correlations','signed_r','Time-course stability')]):
        for col, modality in enumerate(['ieeg','meg']):
            ax = axes[row,col]
            matrix = np.full((n,n),np.nan)
            for record in stability[table].query('modality == @modality').itertuples():
                value = getattr(record,metric)
                matrix[record.iteration_a,record.iteration_b] = value
                matrix[record.iteration_b,record.iteration_a] = value
            im = ax.imshow(np.ma.masked_invalid(matrix),cmap=cmap,vmin=-1,vmax=1)
            ticks = np.unique(np.linspace(0,n-1,min(n,6),dtype=int))
            ax.set(title=f'{modality}: {title}',xlabel='Iteration',ylabel='Iteration',
                   xticks=ticks,xticklabels=ticks+1,yticks=ticks,yticklabels=ticks+1); ax.grid(True)
            fig.colorbar(im,ax=ax,label='Pearson r',shrink=.75)
    roi = stability['roi_stability']
    rois = sorted(roi.roi.dropna().unique())
    for col, (modality,color) in enumerate([('ieeg',MODALITY_COLORS['iEEG']),('meg',MODALITY_COLORS['MEG'])]):
        ax = axes[2,col]
        values = roi.query('modality == @modality').set_index('roi').reindex(rois)
        ax.errorbar(np.arange(len(rois)),values.mean_magnitude,yerr=values[ROI_ERROR],fmt='o',capsize=3,color=color)
        ax.set(title=f'{modality}: bilateral ROI magnitudes',ylabel='Mean |weight|',
               xticks=np.arange(len(rois)),xticklabels=rois)
        ax.tick_params(axis='x',labelrotation=90,labelsize=ROI_LABEL_SIZE); ax.grid(True)
    cross = stability['roi_cross_correlations']
    for col,(metric,title) in enumerate([('mean_r','Mean signed channel-pair r'),('sd_r','SD across channel-pair r')]):
        ax = axes[3,col]
        if rois:
            matrix = cross.pivot(index='ieeg_roi',columns='meg_roi',values=metric).reindex(index=rois,columns=rois)
            colors = plt.get_cmap('RdBu_r' if col == 0 else 'viridis').copy(); colors.set_bad('lightgray')
            im = ax.imshow(np.ma.masked_invalid(matrix.to_numpy()),cmap=colors,
                           vmin=-1 if col == 0 else 0,vmax=1 if col == 0 else np.sqrt(2))
            ax.set(title=title,xlabel='MEG bilateral ROI',ylabel='iEEG bilateral ROI',xticks=np.arange(len(rois)),
                   xticklabels=rois,yticks=np.arange(len(rois)),yticklabels=rois)
            ax.tick_params(axis='x',labelrotation=90,labelsize=ROI_LABEL_SIZE)
            ax.tick_params(axis='y',labelsize=ROI_LABEL_SIZE)
            fig.colorbar(im,ax=ax,shrink=.75)
        else:
            ax.text(.5,.5,'No labeled ROIs',ha='center',transform=ax.transAxes)
        ax.grid(True)
    fig.suptitle(f'{STABILITY_MODEL} · {SHUFFLE} · component {COMPONENT} · {n} iterations\n'
                 f'Fixed electrode slots; ROI error: {ROI_ERROR}; gray = undefined',fontsize=13)
    plt.show()


## Subspace overlap

Principal-angle overlap permits rotations within the retained space. Temporal comparisons use shared time points; spatial comparisons use forward patterns mapped to the fixed electrode slots. Dashed lines are full-data descriptions; solid lines are held-out fold medians, with fold minimum–maximum shading. These ranges are not confidence intervals.


In [ ]:
FIGSIZE = (13, 4)

if tables is not None:
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)
    for ax, space in zip(axes,['temporal_scores','spatial_patterns']):
        for name in selected_models:
            data = tables['overlap'].query('model == @name and space == @space and k <= @N_COMPONENTS')
            full = data.query("partition == 'in_sample'").sort_values('k')
            test = data.query("partition == 'test'").groupby('k').overlap.agg(['median','min','max'])
            ax.plot(full.k,full.overlap,'--',color=MODEL_COLORS[name],label=name+' all data')
            ax.plot(test.index,test['median'],'o-',color=MODEL_COLORS[name],label=name+' test')
            ax.fill_between(test.index,test['min'],test['max'],color=MODEL_COLORS[name],alpha=.15)
        ax.set(title=space,xlabel='Components',ylabel='Subspace overlap',ylim=(0,1.02)); ax.grid(True); ax.legend(fontsize=8)
    plt.show()


## Which model gives the closest components?

Two complementary comparisons, always at the same number of retained components:

1. **Native cross-modal matching:** `native_matched_abs_r` pairs iEEG and MEG components within each model to maximise mean absolute Pearson correlation. `native_paired_abs_r` keeps the model's own component pairs. `rank_native_matching` ranks the first measure.
2. **Fixed iEEG-PCA reference:** use each model's **MEG-only** scores to reproduce the same independent iEEG PCA time courses. `ieeg_variance_captured`, `subspace_overlap`, and `matched_abs_r` are exactly the metrics from the coverage notebook. `rank_ieeg_reference` ranks variance captured from the retained reference PCs. This fraction is different from reconstruction variance explained in all original iEEG channels.

A model can achieve high native matching by concentrating on a small common signal and leaving much of either dataset unexplained. Read matching alongside reconstruction variance and shared cross-covariance capture. 

In [ ]:
INPUT_DIR = ROOT / 'out' / 'covariance'
BASE_CONFIG_FILE = INPUT_DIR / 'config.json'
OUTPUT_CONFIG_FILE = INPUT_DIR / 'compare_modality_config.json'
RECOMPUTE = False

from vs1_run_coverage import analysis_directory, SCHEMA_VERSION
from vs2_run_cov_models import load_models
from vs2_0_cov_models_utils import compare_cov_models
from vs2_0_compare_models import correlation_matrix
covariance_models = None
if BASE_CONFIG_FILE.exists():
    covariance_config = json.loads(BASE_CONFIG_FILE.read_text())
    covariance_config.update(meg_kind=MEG_KIND,block_scaling=BLOCK_SCALING,n_components=N_COMPONENTS,seed=SEED)
    OUTPUT_CONFIG_FILE.write_text(json.dumps(covariance_config,indent=2))
    command = ['python','-u',str(ROOT/'vs2_run_cov_models.py'),'--config',str(OUTPUT_CONFIG_FILE)]
    if RECOMPUTE: command.append('--recompute')
    print(shlex.join(command))
    folder = analysis_directory(covariance_config,'models')
    marker = folder/'RUN_COMPLETE.json'
    if not RECOMPUTE and marker.exists() and json.loads(marker.read_text()) == dict(covariance_config,schema_version=SCHEMA_VERSION):
        all_covariance_models = load_models(folder)
        # Separate PCA remains the fixed reference even when it is not selected for display.
        comparison, component_pairs, reference_pairs = compare_cov_models(
            all_covariance_models,n_components=range(1,N_COMPONENTS+1),plot=False)
        comparison = comparison[comparison.model.isin(selected_models)]
        covariance_models = {name:all_covariance_models[name] for name in selected_models}
        display(comparison)
        display(component_pairs[component_pairs.model.isin(selected_models)])
else:
    print('Set the raw-data configuration in the covariance-model notebook first.')
if covariance_models is None: print('Load matching saved models after running the printed command.')


In [ ]:
FIGSIZE = (15, 4)

if covariance_models is not None:
    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
    for ax, metric in zip(axes,['ieeg_variance_captured','subspace_overlap','matched_abs_r']):
        for name, group in comparison.groupby('model',sort=False):
            ax.plot(group.k,group[metric],'o-',color=MODEL_COLORS[name],label=name)
        ax.set(title=metric,xlabel='Components',ylabel='Similarity',ylim=(0,1.02)); ax.grid(True)
    axes[-1].legend(); plt.show()


In [ ]:
DISPLAY_COMPONENTS = min(5,N_COMPONENTS)

if covariance_models is not None:
    k = DISPLAY_COMPONENTS
    fig, axes = plt.subplots(1,len(covariance_models),figsize=(5*len(covariance_models),4),squeeze=False,constrained_layout=True)
    for ax,(name,model) in zip(axes.flat,covariance_models.items()):
        matrix = correlation_matrix(model.ieeg_scores[:,:k],model.meg_scores[:,:k])
        im = ax.imshow(matrix,vmin=-1,vmax=1,cmap='RdBu_r')
        for (i,j),value in np.ndenumerate(matrix): ax.text(j,i,f'{value:.2f}',ha='center',va='center')
        ax.set(title=name,xlabel='MEG component',ylabel='iEEG component',xticks=np.arange(k),xticklabels=np.arange(1,k+1),
               yticks=np.arange(k),yticklabels=np.arange(1,k+1)); ax.grid(True)
        fig.colorbar(im,ax=ax,label='Signed Pearson r')
    plt.show()


## Transformation generalization

Fit identity, orthogonal, affine, and quadratic maps in both directions using training means, one RMS scale per modality, and a fixed ridge penalty. Apply each map unchanged to held-out responses. NRMSE below one improves on the training-mean baseline; Q² = 1 − NRMSE². Spatial transformations use mapped electrode slots; temporal transformations use shared time points. Dashed full-data curves are descriptive. Shading shows the fold range.


In [ ]:
FIGSIZE = (14, 9)
TRANSFORMATIONS = ['identity','orthogonal','affine','quadratic']

if tables is not None:
    fig, axes = plt.subplots(2,2,figsize=FIGSIZE,constrained_layout=True)
    for row, space in enumerate(['temporal_scores','spatial_patterns']):
        for col, direction in enumerate(['ieeg_to_meg','meg_to_ieeg']):
            ax = axes[row,col]
            for name in selected_models:
                data = tables['alignment'].query('model == @name and space == @space and direction == @direction and k == @N_COMPONENTS')
                full = data.query("partition == 'in_sample'").set_index('complexity').reindex(TRANSFORMATIONS)
                test = data.query("partition == 'test'").groupby('complexity').nrmse.agg(['median','min','max']).reindex(TRANSFORMATIONS)
                ax.plot(range(4),full.nrmse,'--',color=MODEL_COLORS[name],label=name+' all data')
                ax.plot(range(4),test['median'],'o-',color=MODEL_COLORS[name],label=name+' test')
                ax.fill_between(range(4),test['min'],test['max'],color=MODEL_COLORS[name],alpha=.15)
            ax.axhline(1,color='gray',linestyle=':')
            ax.set(title=f'{space}: {direction}',xticks=range(4),xticklabels=TRANSFORMATIONS,ylabel='NRMSE')
            ax.grid(True); ax.legend(fontsize=8)
    fig.suptitle(f'Transformation generalization: {N_COMPONENTS} components')
    plt.show()
